### Delta log P area under the ablation curve

### Todo
- [ ] Add benchmarks on Qwen


In [1]:
import json
import pandas as pd


# Load results
with open("../results/master_results.json") as f:
    data = json.load(f)
     
# Convert to DataFrame (each experiment = row, metrics = columns)
df = pd.DataFrame(data).T
df.index.name = "experiment"
df = df.reset_index()

# Display as table
df

,experiment,top_k_fraction,avg_delta,variance_delta,sem_delta,accuracy,num_correct,total,mean_rank,mean_ranking_pct,sem_mean_ranking_pct,n_with_rank,within_top5_pct_count,fraction_within_top5_pct
0,Llama-3.2-3B__Temperature_lambada_top0.05,0.05,-4.954160,20.151400,0.259174,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-3B__Temperature_lambada_top0.1,0.10,-6.823024,18.421273,0.247799,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-3B__Temperature_lambada_top0.2,0.20,-8.917719,18.778210,0.250188,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.2-3B__Semantic_lambada_top0.05,0.05,-5.050690,18.781744,0.250212,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Llama-3.2-3B__Semantic_lambada_top0.1,0.10,-6.770932,18.357299,0.247368,0.726667,218.0,300.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,gemma-3-1b-pt__Temperature_lmbd1000_top0.1,0.10,-7.130822,37.048892,0.192481,0.391000,391.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
243,gemma-3-1b-pt__Temperature_lmbd1000_top0.2,0.20,-10.467341,33.739090,0.183682,0.391000,391.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
244,gemma-3-1b-pt__Semantic_lmbd1000_top0.05,0.05,-4.034382,30.429380,0.174440,0.391000,391.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN
245,gemma-3-1b-pt__Semantic_lmbd1000_top0.1,0.10,-6.745164,37.013653,0.192389,0.391000,391.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Filter lambada experiments and pivot by drop fraction
# lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower() and "3B" in k}
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Llama-3.2-1B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_16" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k16"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)


    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)


    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Llama-3.2-1B lmbd1000")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")

lambada_table

Llama-3.2-1B lmbd1000
Prediction accuracy: 0.699


method,5%,10%,20%
random,-0.71 ± 0.06,-1.27 ± 0.08,-2.96 ± 0.11
Integrated Grads,-4.32 ± 0.14,-7.07 ± 0.15,-9.14 ± 0.13
Input x Grad,-6.69 ± 0.15,-8.30 ± 0.15,-9.89 ± 0.13
Semantic Scope,-6.67 ± 0.15,-8.47 ± 0.15,-9.91 ± 0.13
Temperature Scope,-6.79 ± 0.15,-8.66 ± 0.14,-9.94 ± 0.13
Fisher Scope k16,-6.94 ± 0.15,-8.64 ± 0.14,-10.12 ± 0.13
Fisher Scope k4,-6.91 ± 0.15,-8.62 ± 0.14,-10.06 ± 0.13
Fisher Scope k1,-6.86 ± 0.15,-8.59 ± 0.14,-10.09 ± 0.12


In [3]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Llama-3.2-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_16" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k16"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
        
    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Llama-3.2-3B lmbd1000")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")
lambada_table

Llama-3.2-3B lmbd1000
Prediction accuracy: 0.775


method,5%,10%,20%
random,-0.62 ± 0.06,-1.17 ± 0.08,-2.51 ± 0.10
Integrated Grads,-1.35 ± 0.08,-3.75 ± 0.12,-7.14 ± 0.14
Input x Grad,-4.95 ± 0.14,-7.20 ± 0.14,-9.11 ± 0.13
Semantic Scope,-5.38 ± 0.14,-7.44 ± 0.14,-9.39 ± 0.14
Temperature Scope,-5.40 ± 0.15,-7.48 ± 0.15,-9.53 ± 0.14
Fisher Scope k1,-5.46 ± 0.14,-7.52 ± 0.14,-9.48 ± 0.13
Fisher Scope k4,-5.39 ± 0.14,-7.65 ± 0.14,-9.57 ± 0.13
Fisher Scope k16,-5.33 ± 0.14,-7.61 ± 0.14,-9.60 ± 0.13


In [4]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "IWSLT2017DE_EN" in k and "Llama-3.2-1B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)

    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
               
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
                     
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Llama-3.2-1B IWSLT2017DE_EN")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")
lambada_table

Llama-3.2-1B IWSLT2017DE_EN
Prediction accuracy: 0.726


method,5%,10%,20%
random,-0.57 ± 0.04,-1.57 ± 0.08,-3.31 ± 0.10
Integrated Grads,-3.90 ± 0.13,-5.89 ± 0.13,-7.91 ± 0.11
Input x Grad,-5.16 ± 0.13,-6.80 ± 0.13,-8.08 ± 0.12
Semantic Scope,-5.41 ± 0.14,-6.86 ± 0.13,-8.16 ± 0.11
Temperature Scope,-5.43 ± 0.13,-7.15 ± 0.12,-8.42 ± 0.11
Fisher Scope k4,-5.62 ± 0.13,-7.12 ± 0.12,-8.32 ± 0.11
Fisher Scope k1,-5.55 ± 0.13,-7.04 ± 0.12,-8.26 ± 0.11


In [5]:

print(lambada_table.to_markdown())

|                   | 5%           | 10%          | 20%          |
|:------------------|:-------------|:-------------|:-------------|
| random            | -0.57 ± 0.04 | -1.57 ± 0.08 | -3.31 ± 0.10 |
| Integrated Grads  | -3.90 ± 0.13 | -5.89 ± 0.13 | -7.91 ± 0.11 |
| Input x Grad      | -5.16 ± 0.13 | -6.80 ± 0.13 | -8.08 ± 0.12 |
| Semantic Scope    | -5.41 ± 0.14 | -6.86 ± 0.13 | -8.16 ± 0.11 |
| Temperature Scope | -5.43 ± 0.13 | -7.15 ± 0.12 | -8.42 ± 0.11 |
| Fisher Scope k4   | -5.62 ± 0.13 | -7.12 ± 0.12 | -8.32 ± 0.11 |
| Fisher Scope k1   | -5.55 ± 0.13 | -7.04 ± 0.12 | -8.26 ± 0.11 |


In [6]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "IWSLT2017DE_EN" in k and "Llama-3.2-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                     
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
           
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    

    
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
   
      
    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

 
    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Llama-3.2-3B IWSLT2017DE_EN")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")
lambada_table

Llama-3.2-3B IWSLT2017DE_EN
Prediction accuracy: 0.754


method,5%,10%,20%
random,-0.45 ± 0.04,-0.95 ± 0.06,-2.08 ± 0.09
Integrated Grads,-1.14 ± 0.08,-3.18 ± 0.11,-6.19 ± 0.13
Temperature Scope,-3.14 ± 0.12,-4.70 ± 0.13,-6.59 ± 0.13
Input x Grad,-3.31 ± 0.11,-4.79 ± 0.12,-6.52 ± 0.13
Semantic Scope,-3.33 ± 0.12,-4.81 ± 0.13,-6.64 ± 0.13
Fisher Scope k4,-3.77 ± 0.12,-5.16 ± 0.13,-6.67 ± 0.13
Fisher Scope k1,-3.56 ± 0.11,-4.99 ± 0.13,-6.67 ± 0.13


In [7]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "IWSLT2017DE_EN" in k and "Qwen2.5-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)

    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
               
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

                             
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Qwen2.5-3B IWSLT2017DE_EN")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")
lambada_table

Qwen2.5-3B IWSLT2017DE_EN
Prediction accuracy: 0.732


method,5%,10%,20%
random,-0.55 ± 0.08,-1.14 ± 0.11,-2.06 ± 0.12
Integrated Grads,-4.96 ± 0.19,-7.12 ± 0.19,-9.05 ± 0.17
Input x Grad,-4.93 ± 0.19,-7.13 ± 0.19,-9.09 ± 0.18
Semantic Scope,-3.87 ± 0.19,-6.31 ± 0.20,-8.81 ± 0.19
Fisher Scope k4,-5.54 ± 0.20,-7.48 ± 0.19,-9.42 ± 0.17
Fisher Scope k1,-5.30 ± 0.14,-7.40 ± 0.14,-9.20 ± 0.12
Temperature Scope,-5.00 ± 0.21,-7.62 ± 0.19,-10.10 ± 0.16


In [8]:
# Filter lambada experiments and pivot by drop fraction
# lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower() and "3B" in k}
lambada_data = {k: v for k, v in data.items() if "lmbd1000" in k.lower() and "Qwen2.5-3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_4" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k4"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    
    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Fisher_k_16" in k and frac_str in k and "random_drop" not in k), None)
    row["Fisher Scope k16"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
        
    sem_key = next((k for k in lambada_data if "Fisher_k_1" in k and frac_str in k and "Fisher_k_16" not in k), None)
    row["Fisher Scope k1"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
            
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)


    
    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
print("Qwen2.5-3B")
print(f"Prediction accuracy: {next(iter(lambada_data.values()))['accuracy']}")
lambada_table

Qwen2.5-3B
Prediction accuracy: 0.725


method,5%,10%,20%
random,-0.75 ± 0.07,-1.43 ± 0.09,-3.35 ± 0.13
Semantic Scope,-6.76 ± 0.15,-8.56 ± 0.15,-10.50 ± 0.14
Integrated Grads,-7.11 ± 0.15,-8.88 ± 0.15,-10.84 ± 0.14
Input x Grad,-7.11 ± 0.15,-8.93 ± 0.15,-10.90 ± 0.14
Fisher Scope k4,-7.38 ± 0.15,-9.13 ± 0.14,-10.93 ± 0.13
Fisher Scope k16,-7.33 ± 0.15,-9.13 ± 0.14,-10.93 ± 0.13
Fisher Scope k1,-7.26 ± 0.15,-9.07 ± 0.14,-10.97 ± 0.13
Temperature Scope,-8.25 ± 0.15,-10.31 ± 0.14,-11.48 ± 0.12


### Most Influential Token

In [9]:
# Filter entries with "loo_rank" (influence ranking vs LOO comparison)
loo_rank_data = {k: v for k, v in data.items() if "loo_rank" in k and "Llama-3.2-1B" in k}

# Map raw method names to display names (same as lambada table)
METHOD_DISPLAY = {
    "random": "random",
    "temperature": "Temperature Scope",
    "gradient_x_input": "Input x Grad",
    "semantic": "Semantic Scope",
    "ig": "Integrated Grads",
    "fisher_k4": "Fisher Scope k4",
    
}

# Build lookup: label key → mean_ranking_pct ± SEM string
def fmt_loo_val(entry):
    mean_pct = entry.get("mean_ranking_pct")
    sem_pct = entry.get("sem_mean_ranking_pct")
    if mean_pct is not None and sem_pct is not None:
        return f"{mean_pct:.1f} ± {sem_pct:.1f}%"
    elif mean_pct is not None:
        return f"{mean_pct:.1f}%"
    return None

method_to_val = {}
for label, entry in loo_rank_data.items():
    parts = label.split("__")
    raw_method = parts[1].replace("_lambada_loo_rank", "") if len(parts) >= 2 else label
    display_method = METHOD_DISPLAY.get(raw_method, raw_method)
    method_to_val[display_method] = fmt_loo_val(entry)

# Build table with same row order as lambada: random, Temperature Scope, Semantic Scope, Gradient Input
# row_order = ["random", "Semantic Scope", "Gradient Input", "Temperature Scope"]
row_order = ["random","Integrated Grads", "Temperature Scope", "Semantic Scope", "Input x Grad", "Fisher Scope k4"]
loo_table = pd.DataFrame(
    {"average ranking": [method_to_val.get(m) for m in row_order]},
    index=row_order,
)


loo_table.index.name = "method"
print('Llama-3.2-1B')
loo_table



Llama-3.2-1B


,average ranking
method,
random,48.9 ± 0.9%
Integrated Grads,29.5 ± 1.0%
Temperature Scope,8.0 ± 0.4%
Semantic Scope,6.6 ± 0.4%
Input x Grad,5.7 ± 0.3%
Fisher Scope k4,5.4 ± 0.3%


In [10]:
# Filter entries with "loo_rank" (influence ranking vs LOO comparison)
loo_rank_data = {k: v for k, v in data.items() if "loo_rank" in k and "Llama-3.2-3B" in k}

# Map raw method names to display names (same as lambada table)
METHOD_DISPLAY = {
    "random": "random",
    "temperature": "Temperature Scope",
    "gradient_x_input": "Input x Grad",
    "semantic": "Semantic Scope",
    "ig": "Integrated Grads",
    "fisher_k4": "Fisher Scope k4",
    
}

# Build lookup: label key → mean_ranking_pct ± SEM string
def fmt_loo_val(entry):
    mean_pct = entry.get("mean_ranking_pct")
    sem_pct = entry.get("sem_mean_ranking_pct")
    if mean_pct is not None and sem_pct is not None:
        return f"{mean_pct:.1f} ± {sem_pct:.1f}%"
    elif mean_pct is not None:
        return f"{mean_pct:.1f}%"
    return None

method_to_val = {}
for label, entry in loo_rank_data.items():
    parts = label.split("__")
    raw_method = parts[1].replace("_lambada_loo_rank", "") if len(parts) >= 2 else label
    display_method = METHOD_DISPLAY.get(raw_method, raw_method)
    method_to_val[display_method] = fmt_loo_val(entry)

# Build table with same row order as lambada: random, Temperature Scope, Semantic Scope, Gradient Input
# row_order = ["random", "Semantic Scope", "Gradient Input", "Temperature Scope"]
row_order = ["random","Integrated Grads", "Temperature Scope", "Input x Grad", "Semantic Scope", "Fisher Scope k4"]
loo_table = pd.DataFrame(
    {"average ranking": [method_to_val.get(m) for m in row_order]},
    index=row_order,
)
loo_table.index.name = "method"
print('Llama-3.2-3B')
loo_table

Llama-3.2-3B


,average ranking
method,
random,48.5 ± 0.9%
Integrated Grads,42.8 ± 0.8%
Temperature Scope,12.3 ± 0.6%
Input x Grad,11.7 ± 0.5%
Semantic Scope,10.2 ± 0.5%
Fisher Scope k4,9.4 ± 0.5%
